# Score-Based Models, SDEs, and Probability Flow ODEs

The discrete diffusion picture can be pushed further by asking what happens when the time step becomes very small and the number of steps becomes very large. In that regime, the noising process is better described as a **stochastic differential equation**, and the reverse dynamics become a continuous-time denoising flow. This perspective is not merely a mathematical curiosity. It clarifies why noise prediction, score prediction, stochastic sampling, and deterministic sampling are closely related rather than separate ideas.

The continuous-time viewpoint is developed in layers. First comes the intuitive difference between ordinary and stochastic differential equations. Then come score functions and the reason they are the right objects to learn along a family of perturbed data distributions. After that come the forward SDE, the reverse-time SDE, and the probability flow ODE. Finally, these ideas reconnect with denoising score matching and with the VP and VE families used in modern diffusion models.


Diffusion models can be described in at least three compatible languages: **denoising regression**, **score estimation**, and **stochastic dynamics**. None of these viewpoints is redundant. The denoising view makes optimization intuitive, the score view explains what function is actually being learned, and the SDE/ODE view explains how generation can proceed through either stochastic or deterministic evolution.

Continuous-time notation can look intimidating at first, but the underlying structure is straightforward. The main point is to make clear why continuous-time diffusion models are mathematically natural and why their practical algorithms are not disconnected hacks.


```{important}
The continuous-time viewpoint is mainly a **change of language**. The same diffusion ideas are being rewritten in terms of scores, SDEs, and ODEs.
```


## An Intuitive Digression on ODEs and SDEs

Before entering the score-based formalism, it is useful to pause for a conceptual distinction. An ordinary differential equation describes deterministic evolution. If a state $\boldsymbol{x}(t)$ follows
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{f}(\boldsymbol{x}(t), t),
:::
then the vector field $\boldsymbol{f}$ tells us the instantaneous velocity at every point and time. Once the initial condition is fixed, the trajectory is fixed as well. Deterministic transport models, including flow matching and continuous normalizing flows, live in this world.

A stochastic differential equation adds a random perturbation at every instant. In informal notation,
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w},
:::
where $\boldsymbol{w}$ is a Brownian motion. The first term is still a drift that pushes the state in a coherent direction. The second term injects infinitesimal Gaussian randomness. As a result, the evolution of a single trajectory is random even when the initial condition is fixed, but the distribution of trajectories follows a well-defined law. This is precisely the level at which diffusion models operate: they are less concerned with one exact path than with how probability mass evolves over time.

A useful picture is to think of an **ODE** as a flow of particles in a velocity field and an **SDE** as a flow of particles that are simultaneously advected and continuously shaken by noise. In image generation, the noising process is modeled by an SDE because the data distribution must spread gradually into a simpler reference distribution. The reverse generative process then learns how to undo this stochastic spreading.


One more intuition helps. In an ODE, nearby particles that start at the same point and follow the same vector field remain synchronized because no external randomness separates them. In an SDE, even particles with the same initial condition can diverge because **Brownian noise** perturbs them differently. This is exactly why an SDE is such a good language for noising: it describes not just a deterministic deformation of the data manifold, but an actual spreading of probability mass into surrounding space.

This distinction becomes practically important when comparing **stochastic reverse sampling** with **deterministic probability flow ODE** sampling. The two procedures can share the same family of marginals even though one uses randomness and the other does not.


## Scores and Perturbed Data Distributions

Let $p_t$ denote a time-dependent probability density on image space. The **score** of this density is defined by
:::{math}
\nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x}).
:::
The score points in the direction of increasing log-density, so it can be interpreted as a local vector that tells us how the density rises around the point $\boldsymbol{x}$. Unlike the density itself, the score is invariant under unknown normalization constants, which is one reason it is attractive in high-dimensional modeling.

In score-based generative modeling, the aim is not to learn only the score of the clean data distribution. Instead, one learns the scores of a whole family of perturbed distributions obtained by corrupting data with Gaussian noise of varying strength. This is crucial. The clean data distribution can be highly singular or concentrated near a thin manifold, making its score unstable or difficult to estimate directly. Once noise is added, the perturbed density becomes smoother and more regular, and the associated score becomes a better learning target.

It is helpful to interpret the score geometrically. The vector $\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})$ points toward regions where the current marginal density is higher. If a noisy sample lies in a low-density direction away from typical data, the score points back toward more likely structure. This is why **score learning** and **denoising** are so tightly linked. A good score field is, locally, a map of how probability mass should be pulled back toward realistic configurations.

One can think of a noisy face image here. If random corruption pushes one eye slightly out of alignment or inserts grain in the cheek region, the score near that corrupted point points toward configurations that look more like typical faces. The score is therefore not an abstract derivative with no operational meaning. It is a directional hint about how to nudge a sample back toward the data distribution.

There is also a normalization advantage here. Learning the density itself in high dimension is hard partly because the absolute scale of the density matters and normalizing constants are expensive. The score removes that obstacle by differentiating the log-density. One no longer needs the partition function explicitly. This is one of the main reasons score-based methods became so attractive theoretically.


## Denoising Score Matching

Suppose we observe noisy samples of the form
:::{math}
\widetilde{\boldsymbol{x}} = \boldsymbol{x} + \sigma \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}),
:::
where $\boldsymbol{x} \sim p_{gt}$. Let $p_\sigma$ denote the density of the noisy variable $\widetilde{\boldsymbol{x}}$. A neural network $\boldsymbol{s}_\theta(\widetilde{\boldsymbol{x}}, \sigma)$ can be trained to approximate the score $\nabla_{\widetilde{\boldsymbol{x}}} \log p_\sigma(\widetilde{\boldsymbol{x}})$. One particularly useful objective is denoising score matching, which avoids explicit evaluation of the perturbed density.

```{prf:theorem} Optimal target in denoising score matching
:label: thm-dsm-target

Consider the objective
:::{math}
\mathcal{L}_{DSM}(\theta)
=
\mathbb{E}_{\boldsymbol{x}, \widetilde{\boldsymbol{x}} | \boldsymbol{x}}
\left[
    \left\|
        \boldsymbol{s}_\theta(\widetilde{\boldsymbol{x}}, \sigma)
        -
        \nabla_{\widetilde{\boldsymbol{x}}}
        \log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
    \right\|_2^2
\right],
:::
where $q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x}) = \mathcal{N}(\widetilde{\boldsymbol{x}}; \boldsymbol{x}, \sigma^2 \boldsymbol{I})$. The minimizer of this objective is
:::{math}
\boldsymbol{s}_\theta^\star(\widetilde{\boldsymbol{x}}, \sigma)
=
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
```

```{prf:proof}
For any fixed noisy observation $\widetilde{\boldsymbol{x}}$, the conditional expectation identity for squared error implies that the optimal predictor is
:::{math}
\mathbb{E}
\left[
    \nabla_{\widetilde{\boldsymbol{x}}}
    \log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
    \,|\,
    \widetilde{\boldsymbol{x}}
\right].
:::
Since
:::{math}
p_\sigma(\widetilde{\boldsymbol{x}})
=
\int q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x}) p_{gt}(\boldsymbol{x})\, d\boldsymbol{x},
:::
differentiating under the integral sign gives
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}} p_\sigma(\widetilde{\boldsymbol{x}})
=
\int
\nabla_{\widetilde{\boldsymbol{x}}} q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
p_{gt}(\boldsymbol{x})\, d\boldsymbol{x}.
:::
Dividing by $p_\sigma(\widetilde{\boldsymbol{x}})$ yields
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}})
=
\int
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
\, p(\boldsymbol{x} | \widetilde{\boldsymbol{x}})\, d\boldsymbol{x},
:::
which is exactly the conditional expectation above. Hence the optimal network equals the score of the perturbed density.
```

Because the Gaussian conditional score is explicit,
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
=
-\frac{\widetilde{\boldsymbol{x}} - \boldsymbol{x}}{\sigma^2},
:::
the objective becomes a tractable regression problem. This is the first major bridge between diffusion modeling and denoising: learning a score field can be turned into learning how corruption relates noisy samples back to clean ones.

This theorem is one of the cleanest examples of a hard generative objective being transformed into an easy regression target. The model is asked to predict a quantity derived from a Gaussian conditional that is known analytically, yet the minimizer recovers the score of the unknown noisy data distribution. In spirit, this mirrors what happens in DDPMs when the variational objective collapses into noise prediction.

The network is not asked to estimate the score of the empirical data distribution directly at zero noise. Instead, it is trained across a continuum or grid of noise scales where the distributions are smoother. The difficult object is therefore approached through a family of easier auxiliary problems.


## Forward SDEs and Their Marginals

In the continuous-time formulation, the noisy data process evolves according to an Itô **SDE**
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w},
:::
where $\boldsymbol{w}$ is standard Brownian motion. The drift $\boldsymbol{f}$ and diffusion amplitude $g$ are chosen so that, as time increases, the distribution of $\boldsymbol{x}(t)$ becomes progressively simpler. One usually wants the terminal law at time $T$ to be close to a standard Gaussian that is easy to sample from.

Two particularly important families are the **variance exploding** and **variance preserving** SDEs. In the **VE-SDE**, the drift vanishes and noise strength grows over time:
:::{math}
d\boldsymbol{x} = \sqrt{\frac{d[\sigma^2(t)]}{dt}}\, d\boldsymbol{w}.
:::
Here the signal itself is not contracted by the drift, but the variance of the perturbation keeps increasing. In the **VP-SDE**, one uses
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt
+
\sqrt{\beta(t)}\, d\boldsymbol{w}.
:::
This process shrinks the signal while injecting noise, so the total variance remains controlled even though the initial information is gradually destroyed. For image generation, the VP family is especially important because it is the continuous-time analogue most closely connected to DDPMs.


The VP versus VE distinction is worth making concrete because the acronyms can otherwise feel like catalog labels. In a VP process, the signal amplitude is explicitly damped by the drift while noise is added, so the original image is gradually forgotten and replaced by a controlled-variance noisy state. In a VE process, the original signal is not contracted by a drift term; instead, the observation is overwhelmed by increasingly large additive noise. Both routes can end near a simple reference law, but they organize the path to that law differently.

This difference affects both theory and implementation. The natural parameterizations of time, the interpretation of the denoiser, and the numerical stiffness of the reverse dynamics can differ between VP and VE constructions. DDPMs sit conceptually much closer to the VP family, whereas the broader score-based literature often discusses both VP and VE as parallel continuous-time noising choices.


## Reverse-Time SDE

The forward SDE explains how to corrupt data. Generation requires the reverse evolution that starts from a simple terminal distribution and moves toward the data distribution. The remarkable fact is that the **reverse-time dynamics** are again governed by an SDE, but with a drift corrected by the **score** of the time-marginal density.


This theorem contains the conceptual heart of score-based diffusion. The reverse drift is not arbitrary. It equals the forward drift corrected by a term that points toward regions of higher density in the current marginal distribution. In other words, the score tells the reverse dynamics how to denoise. The only missing ingredient is that the true score is unknown, so it must be approximated by a neural network $\boldsymbol{s}_\theta(\boldsymbol{x}, t)$ trained from noisy data.

This is the place where the whole diffusion story condenses into one sentence: if you know the score of the time-marginal density at every noise level, then you know how to run the generative process backward. The unknown object in the reverse dynamics is therefore not an arbitrary conditional law but one geometrically meaningful vector field. This is why score estimation is not an auxiliary idea around diffusion; it is the central object that makes reverse sampling possible.

From the viewpoint of scientific understanding, this theorem is also satisfying because it separates modeling from simulation. Modeling means estimating the score field. Simulation means numerically integrating either the reverse SDE or a related deterministic ODE once that field is available. The same trained network can therefore support multiple samplers, which is one reason the diffusion literature contains so much work on fast and improved generation methods.

## Tweedie's Formula

Tweedie's formula makes the denoising interpretation even more explicit. It states that for Gaussian corruption, the posterior mean of the clean signal can be written directly in terms of the score of the noisy distribution.

```{prf:theorem} Tweedie's formula
:label: thm-tweedie

Let
:::{math}
\widetilde{\boldsymbol{x}} = \boldsymbol{x} + \sigma \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}),
:::
and let $p_\sigma(\widetilde{\boldsymbol{x}})$ denote the density of the noisy observation. Then
:::{math}
\mathbb{E}[\boldsymbol{x} | \widetilde{\boldsymbol{x}}]
=
\widetilde{\boldsymbol{x}}
+
\sigma^2
\nabla_{\widetilde{\boldsymbol{x}}}
\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
```

```{prf:proof}
From the Gaussian conditional density,
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
=
\frac{\boldsymbol{x} - \widetilde{\boldsymbol{x}}}{\sigma^2}.
:::
Taking the conditional expectation with respect to $\boldsymbol{x} | \widetilde{\boldsymbol{x}}$ and using the identity proved in the denoising score matching theorem gives
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log p_\sigma(\widetilde{\boldsymbol{x}})
=
\frac{\mathbb{E}[\boldsymbol{x} | \widetilde{\boldsymbol{x}}] - \widetilde{\boldsymbol{x}}}{\sigma^2}.
:::
Rearranging yields the formula.
```

Tweedie's formula shows that a score estimator is implicitly a denoiser. Once the score of the noisy distribution is known, one can recover the posterior mean of the clean sample. This is one reason diffusion, score matching, and denoising are so tightly intertwined in the literature. They are not separate intuitions accidentally leading to similar algorithms; they are different views of the same mathematical identity.

This identity translates an abstract differential quantity into an estimator of a familiar object. Tweedie's formula says that the score tells us how to correct a noisy observation toward the posterior mean of the clean signal. In other words, the score is not just a geometric arrow field; it is directly actionable for denoising.

There is also a clear continuity with the discrete diffusion picture. There, the network predicts noise and from that prediction one can reconstruct an estimate of $\boldsymbol{x}_0$. Here, the score plays the analogous role in continuous time. The mathematical packages differ, but the operational meaning is the same: infer how the current noisy sample should move toward cleaner structure.


## Probability Flow ODE

The reverse-time SDE is stochastic, but remarkably there exists a deterministic ODE with exactly the same time marginals. This is the **probability flow ODE**. It provides a bridge from stochastic diffusion models to deterministic transport models.


This theorem is extremely important conceptually. It says that the same score network can support both stochastic sampling, through the reverse SDE, and deterministic sampling, through the probability flow ODE. In practice, the ODE viewpoint also enables likelihood-related calculations and creates a clear conceptual link with continuous normalizing flows and, later, with flow matching.

For many readers, this is the most surprising result in the discussion. One may expect that if the forward corruption uses stochastic noise, then the reverse generative process must also be stochastic. The probability flow ODE says otherwise: there exists a deterministic evolution that shares the same one-time marginals as the stochastic reverse diffusion. This does not mean that individual trajectories match. It means that the distribution at each time is the same.

This is one of the clearest conceptual bridges between diffusion and flow matching. A score-based diffusion model can be viewed not only as a stochastic denoiser but also as a learned transport system.


For many readers, this is the most surprising theorem in the chapter. One may expect that if the forward corruption used stochastic noise, then the reverse generative process must also be stochastic. The probability flow ODE says otherwise: there exists a deterministic evolution that shares the same one-time marginals as the stochastic reverse diffusion. This does not mean that individual trajectories match. It means that the distribution at each time is the same.

This observation is one of the clearest conceptual bridges between diffusion and later flow matching ideas. A score-based diffusion model can be viewed not only as a stochastic denoiser but also as a learned transport system. Once students understand this, the transition to deterministic transport methods becomes far smoother.

## VP-SDE, VE-SDE, and the Relation with DDPMs

We can now position the major continuous-time families more clearly. In the **VP-SDE**,
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt
+
\sqrt{\beta(t)}\, d\boldsymbol{w},
:::
the mean signal decays exponentially while noise is injected continuously. This is the continuous analogue of the discrete Gaussian noising process used in DDPMs, and for this reason it is the main focus of these notes. In the **VE-SDE**, the drift is zero and the noise standard deviation grows directly with time. Both constructions produce families of perturbed densities whose scores can be learned, but they differ in numerical behavior and in the geometry of the perturbation path.

From the discrete viewpoint, DDPM training often predicts the noise $\boldsymbol{\varepsilon}$. From the continuous-time viewpoint, the more intrinsic object is the **score** $\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})$. The two parameterizations are equivalent up to analytic transformations determined by the perturbation law. This is why discrete diffusion, denoising score matching, and continuous-time score-based modeling form one coherent family rather than disconnected algorithms.


This is also the right place to mention latent diffusion from the continuous-time viewpoint. Once score or noise prediction is understood as learning a reverse evolution in some representation space, there is no requirement that this space be the raw pixel grid. If an autoencoder provides a compressed latent representation in which semantics are preserved reasonably well, then the same VP- or VE-style reasoning can be applied there. The practical gain is enormous because the differential evolution is solved in a lower-dimensional and often smoother space. The conceptual cost is that one must now trust the latent representation as part of the generative model.

## Continuous-Time Implementation View

The same objects that appeared in the mathematics can be expressed as ordinary tensor computations: a time-dependent noise scale, a **score field**, a **reverse-SDE step**, and a **probability-flow ODE step**. The goal of the code below is not to train a full modern score-based model from scratch, but to make the mathematical objects look operational rather than ceremonial.

The translation happens at the level of the continuous-time perturbation law and the sampler update rules.


## Continuous-Time Implementation View

At this point, the continuous-time picture should not remain only a theorem sheet. The same objects that appeared in the mathematics can be expressed as ordinary tensor computations: a time-dependent noise scale, a **score field**, a **reverse-SDE step**, and a **probability-flow ODE step**. The goal of the code below is not to train a full modern score-based model from scratch, but to make the mathematical objects look operational rather than ceremonial.

This mirrors what happened in the discrete DDPM chapter. There, the forward marginal and noise-prediction loss were translated into code directly. Here, the translation happens at the level of the continuous-time perturbation law and the sampler update rules.


In [ ]:
import math
import torch


def linear_beta(t, beta_min=0.1, beta_max=20.0):
    return beta_min + t * (beta_max - beta_min)


def vp_log_mean_coeff(t, beta_min=0.1, beta_max=20.0):
    return -0.25 * (beta_max - beta_min) * t**2 - 0.5 * beta_min * t


def vp_mean_std(x0, t, beta_min=0.1, beta_max=20.0):
    log_mean = vp_log_mean_coeff(t, beta_min=beta_min, beta_max=beta_max)
    mean_coeff = torch.exp(log_mean).view(-1, 1, 1, 1)
    std = torch.sqrt(torch.clamp(1.0 - torch.exp(2.0 * log_mean), min=1e-5)).view(-1, 1, 1, 1)
    return mean_coeff * x0, std


def perturb_vp_data(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    mean, std = vp_mean_std(x0, t)
    xt = mean + std * noise
    return xt, noise, std


def score_from_noise_prediction(predicted_noise, std):
    return -predicted_noise / std


The continuous-time implementation pattern is now easy to read. `perturb_vp_data` plays the same role as `q_sample` in the DDPM setting, except that time is now continuous. If a network predicts the noise, `score_from_noise_prediction` converts that prediction into the score field required by the reverse dynamics.


In [ ]:
@torch.no_grad()
def reverse_sde_step(x, t, dt, score_model, beta_min=0.1, beta_max=20.0):
    beta_t = linear_beta(t, beta_min=beta_min, beta_max=beta_max).view(-1, 1, 1, 1)
    drift = -0.5 * beta_t * x
    score = score_model(x, t)
    reverse_drift = drift - beta_t * score
    noise = torch.randn_like(x)
    return x + reverse_drift * dt + torch.sqrt(torch.clamp(beta_t * abs(dt), min=1e-6)) * noise


@torch.no_grad()
def probability_flow_step(x, t, dt, score_model, beta_min=0.1, beta_max=20.0):
    beta_t = linear_beta(t, beta_min=beta_min, beta_max=beta_max).view(-1, 1, 1, 1)
    drift = -0.5 * beta_t * x
    score = score_model(x, t)
    ode_drift = drift - 0.5 * beta_t * score
    return x + ode_drift * dt


These two update rules are the code-level manifestation of the main theorems. The reverse-time SDE step uses stochastic noise and the full score-corrected drift. The probability flow step removes the stochastic term and halves the score correction, exactly as predicted by the probability flow ODE.


In [ ]:
@torch.no_grad()
def euler_maruyama_sampler(score_model, shape, num_steps=200, device="cpu"):
    x = torch.randn(shape, device=device)
    time_grid = torch.linspace(1.0, 1e-3, num_steps, device=device)
    dt = -(time_grid[0] - time_grid[1]).item()
    for t_scalar in time_grid:
        t = torch.full((shape[0],), t_scalar, device=device)
        x = reverse_sde_step(x, t, dt, score_model)
    return x


@torch.no_grad()
def probability_flow_sampler(score_model, shape, num_steps=200, device="cpu"):
    x = torch.randn(shape, device=device)
    time_grid = torch.linspace(1.0, 1e-3, num_steps, device=device)
    dt = -(time_grid[0] - time_grid[1]).item()
    for t_scalar in time_grid:
        t = torch.full((shape[0],), t_scalar, device=device)
        x = probability_flow_step(x, t, dt, score_model)
    return x


The important lesson is not that these few lines are production samplers. They are not. The important lesson is that once the score field is learned, the rest of continuous-time diffusion really does reduce to numerical integration of either a stochastic or deterministic dynamics. That is exactly the conceptual bridge from score-based diffusion to later flow-based methods.


## Final Perspective

The forward process spreads data toward a simple reference distribution. The score network learns how density changes across that perturbation path. The **reverse-time SDE** converts this learned score into a stochastic sampler, while the **probability flow ODE** converts the same score into a deterministic sampler with matching marginals. **Tweedie's formula** explains why score estimation and denoising are two sides of the same coin.

This picture also makes clear why diffusion and flow matching sit so close to each other conceptually. Both are path-based generative models. The main difference is whether the path is introduced through stochastic perturbation and a score field, or through a directly specified transport field.


```{figure} ../assets/images/sinusoidal_embedding.png
:width: 72%
:align: center

Time-conditioning is a central ingredient in practical diffusion models, since the network must know which noise level or time point it is denoising.
```